# Data preparation.

To get the data ready for modeling the following actions are performed:
- **Feature engineering**
- **Target creation**
- **Chronological split**
- **Scaling**
- **Sliding windows**


### Setup

In [1]:
# --- Notebook setup: make the project's modules (config.py, src/) importable ---
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if "TF_CPP_MIN_LOG_LEVEL" not in os.environ:
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow info/warning logs

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

### Load the daset

During data understanding the apple stock dataset was saved on a pickle file so it can be loaded from it.

In [3]:
""" Open the pickle file and load the apple stock data back into a DataFrame """

import config
import pickle

file_path = config.DATA_DIR / 'apple_data.pkl'
with open(file_path, 'rb') as f:
    apple_df = pickle.load(f)
    print(f"Loaded cleaned Apple dataset from {file_path}")
    print(f"shape (rows, columns): {apple_df.shape}")

Loaded cleaned Apple dataset from G:\Shared drives\Group 7 for AI 570\Stock_Price_DL\data\apple_data.pkl
shape (rows, columns): (4203, 5)


#### Alternatively a data loader code has been prepared to load the dataset from the raw data and get it ready for data preparation.

```python
from src.data_loader import load_stock_data

apple_df = load_stock_data()
```

## Feature engineering

During data understanding the following engineering features were identified:

| Feature | Formula | Description |
|---|---|---|
| `log_return` | $\ln(C_t / C_{t-1})$ | How much the price moves daily |
| `log_volume` | $\ln(1 + V_t)$ | How much trading happened |
| `volume_change` | $\text{log\_volume}_t - \text{log\_volume}_{t-1}$ | trading activity direction: rising or falling |

Rule: a feature for day *t* may only use information available at the close of day *t*.

A helper function `add_features` has been created to add those features to the dataset.

In [4]:
""" Add engineered features to the apple_df DataFrame """
from src.features import add_features

feat = add_features(apple_df)
feat[["Close", "Volume", "log_return", "log_volume", "volume_change"]].head()

Price,Close,Volume,log_return,log_volume,volume_change
2010-01-04,6.400959,493729600,NaN,20.017499,NaN
2010-01-05,6.412026,601904800,0.001727,20.215610,0.198111
2010-01-06,6.310035,552160000,-0.016034,20.129348,-0.086261
2010-01-07,6.298369,477131200,-0.001850,19.983302,-0.146046
2010-01-08,6.340242,447610800,0.006626,19.919435,-0.063867


The first row has `NaN` returns because there is no previous day. Those rows are dropped later.

The switch `use_volume` decides which columns go into the model:

In [5]:
""" Check the feature columns returned by get_feature_columns() with and without volume features."""
from src.features import get_feature_columns

print("use_volume=False ->", get_feature_columns(use_volume=False))
print("use_volume=True  ->", get_feature_columns(use_volume=True))

use_volume=False -> ['log_return']
use_volume=True  -> ['log_return', 'log_volume', 'volume_change']


## Target
The target is tomorrow's value.

For each row *t*, the target is **tomorrow's** log return. `shift(-1)` moves each value one row up, so row *t* receives the value from *t+1*:

```
date (t)     log_return_t    target_t = log_return_(t+1)
day 1        r1              r2
day 2        r2              r3
...
last day     r_last          NaN   <- unknown until tomorrow: the value we predict in deployment
```

The function `build_modeling_frame` adds the engineering features, the prediction target and removes the first and last rows because they are incomplete.

In [6]:
""" Build the modeling frame with features and target, removing incomplete rows.
"""
from src.features import build_modeling_frame

frame = build_modeling_frame(apple_df)   # features + target, incomplete rows removed
frame[["Close", "log_return", "target", "next_close", "target_date"]].head()

Price,Close,log_return,target,next_close,target_date
2010-01-05,6.412026,0.001727,0.001727,6.310035,2010-01-06
2010-01-06,6.310035,-0.016034,-0.016034,6.298369,2010-01-07
2010-01-07,6.298369,-0.001850,-0.001850,6.340242,2010-01-08
2010-01-08,6.340242,0.006626,0.006626,6.284313,2010-01-11
2010-01-11,6.284313,-0.008861,-0.008861,6.212827,2010-01-12


To confirm the modeling frame is ready we verify target is shifted by one day.

In [ ]:
""" Verify alignment of target and next_close with log_return and Close."""
# Confirm each row's target equals the NEXT row's log return
assert np.allclose(frame["target"].to_numpy()[:-1], frame["log_return"].to_numpy()[1:])
# Verify today's close x exp(target) rebuilds tomorrow's close exactly
assert np.allclose(frame["Close"] * np.exp(frame["target"]), frame["next_close"])
print(f"Target alignment verified for {len(frame)} rows.")

AssertionError: 

Next: [`03_modeling.ipynb`](03_modeling.ipynb)